## 1. Install required Python packages

In [ ]:
%pip install langchain langgraph langchain-openai

## 2. Get Endpoints

Retrieve the FQDN of the self-hosted LLM and the Cosmos DB connection details from Terraform outputs.

In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4


## 3. Set Up the LLM Model

Create a `ChatOpenAI` model pointing at the vLLM-compatible endpoint running on Azure Container Apps.

In [4]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens=512
)

# model = ChatOpenAI(
#     base_url=f"{foundry_endpoint}/openai/v1",
#     api_key=foundry_api_key,
#     model=llm_model_deployment_name_chatgpt,
#     streaming=True,
#     max_completion_tokens=512
# )

## 4. Test the Model

Invoke the model with a test prompt to ensure it's working correctly.

In [9]:
from langchain_core.messages import HumanMessage

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I am a large language model, trained by Google. 

If you’re looking for a more detailed breakdown of what that actually means, here is a quick summary of who I am and what I can do:

### 🛠️ What I Am
Think of me as a highly advanced pattern-recognition engine. I have processed a massive amount of text—books, articles, code, and conversations—which allows me to understand the nuances of human language, logic, and creativity. I don't "know" things the way a human does through experience; instead, I predict the most helpful and accurate way to respond based on the vast amount of data I was trained on.

### 🚀 What I Can Do
I am designed to be a versatile digital assistant. Some of my core strengths include:

*   **Writing & Creativity:** I can draft emails, write essays, compose poems, create scripts, or help you brainstorm ideas for a project.
*   **Learning & Synthesis:** I can take complex topics (like quantum physics or how a mortgage works) and explain them in simple terms. I can also

## 5. Use the Model in a LangChain Agent

In [10]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[],
    checkpointer=None,
)

response = agent.stream({"messages": HumanMessage(content="Tell me about yourself")})

async for step in agent.astream(
    {"messages": [HumanMessage(content="Tell me about yourself")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about yourself
================================== Ai Message ==================================

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- analyzing text or data
- tutoring across many subjects

A few useful things to know about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data I was trained on.
- I can be very useful, but I can also make mistakes, so important facts should be verified.
- I don’t “know” things the way humans do; I predict helpful responses from the context I’m given.
- In this chat, I respond through an API-based interface rather than as a person with a life or identity.

If you want, I can also tell you about:
- how I work
- what I’m good at
- my limitatio